# Sentiment Analysis: Predicted vs. Ground Truth
VADER + RoBERTa cross-validation on justification text.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.analysis.data_loading import load_annotations
from src.analysis.sentiment import evaluate_all_models, add_vader_predictions
sns.set_theme(style='whitegrid')

In [ ]:
df = load_annotations()
accuracy_df = evaluate_all_models(df)
accuracy_df.to_csv('../outputs/analysis/human_annotator_agreement.csv', index=False)
accuracy_df.sort_values('f1_macro', ascending=False)

In [ ]:
# Heatmap: F1-macro per (culture × model)
pivot = accuracy_df.pivot(index='culture', columns='model_name', values='f1_macro')
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1)
ax.set_title('Macro-F1 vs. σ₃P₅ Ground Truth')
plt.tight_layout(); plt.savefig('../outputs/figures/02_f1_heatmap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# VADER on justifications
df_vader = add_vader_predictions(df, text_col='justification')
from src.analysis.sentiment import compute_accuracy_metrics
vader_metrics = compute_accuracy_metrics(
    df_vader['ground_truth_sentiment'].tolist(),
    df_vader['vader_sentiment'].tolist()
)
print('VADER on justifications:', vader_metrics)